# Part 3 — Adversarial Attacks: Breaking the Classifier
**Assignment 2 | Responsible & Explainable AI | FAST-NUCES**

Two adversarial attacks implemented from scratch:

1. **Attack 1 — Character-level evasion**: perturb toxic comments with zero-width spaces,
   Unicode homoglyphs, and random character duplication. Measure Attack Success Rate (ASR).

2. **Attack 2 — Label-flipping poisoning**: flip 5% of training labels, retrain from
   `distilbert-base-uncased`, and measure degradation in F1, Accuracy, and FNR.

In [ ]:
#!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
import unicodedata
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset as TorchDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OPERATING_THRESHOLD = 0.4   # From Part 1
print(f'Device: {DEVICE}')

---
## Attack 1 — Character-level Evasion

The `perturb(text)` function applies three transformations:
1. **Zero-width space (U+200B)** inserted every 2–3 characters of each word — invisible to humans, splits subword tokens the model relies on.
2. **Unicode homoglyph substitution** — Cyrillic lookalikes replace Latin letters (a→а, e→е, o→о, i→і, c→с). Tokeniser sees unknown subwords; human reads the same text.
3. **Random character duplication** — 20% of characters in each word are doubled ("hate" → "haate"). Breaks n-gram and subword patterns.

In [ ]:
# ── perturb() implementation ─────────────────────────────────────────────────
ZERO_WIDTH_SPACE = '\u200b'   # invisible to human readers

HOMOGLYPHS = {
    'a': '\u0430',   # Cyrillic а
    'e': '\u0435',   # Cyrillic е
    'o': '\u043e',   # Cyrillic о
    'i': '\u0456',   # Ukrainian і (looks like Latin i)
    'c': '\u0441',   # Cyrillic с
    'p': '\u0440',   # Cyrillic р (looks like Latin p)
    'x': '\u0445',   # Cyrillic х
    'y': '\u0443',   # Cyrillic у
}

def _insert_zwsp(word: str, gap: int = 2) -> str:
    """Insert zero-width space every `gap` characters."""
    return ZERO_WIDTH_SPACE.join(
        word[i:i+gap] for i in range(0, len(word), gap)
    )

def _homoglyph_substitute(word: str) -> str:
    """Replace Latin chars with Cyrillic lookalikes."""
    return ''.join(HOMOGLYPHS.get(ch.lower(), ch) for ch in word)

def _random_dup(word: str, rate: float = 0.20) -> str:
    """Randomly duplicate 20% of characters in a word."""
    out = []
    for ch in word:
        out.append(ch)
        if random.random() < rate:
            out.append(ch)   # duplicate
    return ''.join(out)

def perturb(text: str) -> str:
    """
    Apply all three character-level perturbations to a text string.
    Pipeline per word: zero-width space insertion → homoglyph substitution → random duplication.
    """
    words = text.split()
    result = []
    for word in words:
        w = _insert_zwsp(word, gap=random.choice([2, 3]))
        w = _homoglyph_substitute(w)
        w = _random_dup(w, rate=0.20)
        result.append(w)
    return ' '.join(result)

# Quick sanity check
sample = "I hate you and you should die"
perturbed = perturb(sample)
print(f'Original   ({len(sample)} chars) : {sample}')
print(f'Perturbed  ({len(perturbed)} chars) : {perturbed}')
print(f'Printable representation: {repr(perturbed[:80])}')

In [ ]:
# ── Load model and eval set ───────────────────────────────────────────────────
print('Loading Part 1 model …')
tokenizer = AutoTokenizer.from_pretrained('./best_model')
model     = AutoModelForSequenceClassification.from_pretrained('./best_model')
model.to(DEVICE).eval()

eval_df = pd.read_csv('eval_subset.csv')

def get_probs(texts, batch_size=64):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc   = tokenizer(batch, max_length=128, truncation=True,
                          padding=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()[:, 1]
        all_probs.append(probs)
    return np.concatenate(all_probs)

print('Model and eval data loaded.')

In [ ]:
# ── Attack 1: Sample 500 high-confidence toxic predictions ───────────────────
import os
if os.path.exists('eval_probs.npy'):
    eval_probs_all = np.load('eval_probs.npy')
else:
    eval_probs_all = get_probs(eval_df['comment_text'].tolist())
    np.save('eval_probs.npy', eval_probs_all)

eval_df['prob_toxic'] = eval_probs_all
eval_df['pred']       = (eval_probs_all >= OPERATING_THRESHOLD).astype(int)

# Filter: model predicted toxic AND confidence >= 0.7
attack_pool = eval_df[(eval_df['pred'] == 1) & (eval_df['prob_toxic'] >= 0.7)].copy()
print(f'High-confidence toxic pool: {len(attack_pool):,} comments')

# Sample 500 (or all if fewer available)
n_attack = min(500, len(attack_pool))
attack_sample = attack_pool.sample(n=n_attack, random_state=SEED).reset_index(drop=True)
print(f'Attack sample size: {n_attack}')
print(f'Mean confidence before attack: {attack_sample["prob_toxic"].mean():.4f}')

In [ ]:
# ── Apply perturbation and measure ASR ───────────────────────────────────────
print(f'Perturbing {n_attack} comments …')
attack_sample['perturbed_text'] = attack_sample['comment_text'].apply(perturb)

print('Running inference on perturbed texts …')
perturbed_probs = get_probs(attack_sample['perturbed_text'].tolist())
perturbed_preds = (perturbed_probs >= OPERATING_THRESHOLD).astype(int)

# Attack Success Rate: fraction that are now predicted NON-TOXIC
asr = (perturbed_preds == 0).sum() / n_attack

print('\n' + '='*55)
print('  ATTACK 1 RESULTS: Character-level Evasion')
print('='*55)
print(f'  Attack sample size              : {n_attack}')
print(f'  Mean confidence BEFORE attack   : {attack_sample["prob_toxic"].mean():.4f}')
print(f'  Mean confidence AFTER  attack   : {perturbed_probs.mean():.4f}')
print(f'  Confidence drop                 : {attack_sample["prob_toxic"].mean() - perturbed_probs.mean():.4f}')
print(f'  Comments evaded (pred flipped)  : {(perturbed_preds == 0).sum()}')
print(f'  Attack Success Rate (ASR)       : {asr:.4f}  ({asr*100:.1f}%)')
print('='*55)

In [ ]:
# ── Visualise confidence shift ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of confidence before vs after
axes[0].hist(attack_sample['prob_toxic'], bins=20, alpha=0.7, color='steelblue',
             label=f'Before (mean={attack_sample["prob_toxic"].mean():.3f})', edgecolor='white')
axes[0].hist(perturbed_probs, bins=20, alpha=0.7, color='darkorange',
             label=f'After  (mean={perturbed_probs.mean():.3f})', edgecolor='white')
axes[0].axvline(OPERATING_THRESHOLD, color='red', lw=2, linestyle='--', label=f'Threshold ({OPERATING_THRESHOLD})')
axes[0].set_xlabel('P(Toxic)', fontsize=12); axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Confidence Before vs After Perturbation', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Scatter: before vs after per comment
axes[1].scatter(attack_sample['prob_toxic'], perturbed_probs, alpha=0.3, s=15, c='purple')
axes[1].plot([0,1],[0,1],'k--',lw=1.5,label='No change')
axes[1].axhline(OPERATING_THRESHOLD, color='red', lw=1.5, linestyle=':')
axes[1].axvline(OPERATING_THRESHOLD, color='red', lw=1.5, linestyle=':')
axes[1].set_xlabel('Confidence Before', fontsize=12)
axes[1].set_ylabel('Confidence After', fontsize=12)
axes[1].set_title(f'Per-Comment Confidence Shift (ASR={asr:.2f})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('part3_attack1.png', dpi=150, bbox_inches='tight')
plt.show()

# Show a few examples
print('\nExample perturbed comments:')
examples = attack_sample.head(3)
for _, row in examples.iterrows():
    print(f'  Original  : {row["comment_text"][:80]}…')
    print(f'  Perturbed : {row["perturbed_text"][:80]}…')
    idx = list(attack_sample.index).index(_)
    print(f'  Confidence: {row["prob_toxic"]:.3f} → {perturbed_probs[idx]:.3f}')
    print()

---
## Attack 2 — Label-Flipping Poisoning

**Threat model**: An attacker with write access to the training data pipeline flips 5% of labels before fine-tuning runs. The model then learns a corrupted decision boundary.

**Procedure**:
- Load 100k training subset
- Randomly flip 5% (5,000) of labels: toxic→non-toxic and non-toxic→toxic
- Retrain from `distilbert-base-uncased` (not from Part 1 fine-tuned weights — the attack targets the fine-tuning process itself)
- Evaluate on the **same clean** evaluation set

In [ ]:
# ── Prepare poisoned training set ─────────────────────────────────────────────
train_df = pd.read_csv('train_subset.csv')
print(f'Training set shape: {train_df.shape}')
print(f'Original label distribution: {train_df["label"].value_counts().to_dict()}')

POISON_FRAC = 0.05
n_poison = int(len(train_df) * POISON_FRAC)
print(f'\nPoisoning {n_poison:,} ({POISON_FRAC*100:.0f}%) randomly selected training rows …')

train_poisoned = train_df.copy()
poison_indices = train_poisoned.sample(n=n_poison, random_state=SEED).index

# Flip labels: 1→0, 0→1
train_poisoned.loc[poison_indices, 'label'] = 1 - train_poisoned.loc[poison_indices, 'label']

print(f'Poisoned label distribution: {train_poisoned["label"].value_counts().to_dict()}')
print(f'Flipped: {n_poison:,} labels')

# How many toxic flipped to non-toxic and vice versa?
orig_labels  = train_df.loc[poison_indices, 'label']
new_labels   = train_poisoned.loc[poison_indices, 'label']
print(f'  Toxic→Non-toxic: {(orig_labels == 1).sum()}')
print(f'  Non-toxic→Toxic: {(orig_labels == 0).sum()}')

In [ ]:
# ── Tokenize poisoned training set ────────────────────────────────────────────
class ToxicityDataset(TorchDataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

MODEL_NAME = 'distilbert-base-uncased'
tok_poison = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Tokenizing poisoned training data …')
poison_enc = tok_poison(
    train_poisoned['comment_text'].tolist(),
    max_length=128, truncation=True, padding='max_length', return_tensors=None
)
print('Tokenizing eval data …')
eval_enc = tok_poison(
    eval_df['comment_text'].tolist(),
    max_length=128, truncation=True, padding='max_length', return_tensors=None
)

poison_dataset = ToxicityDataset(poison_enc, train_poisoned['label'].tolist())
eval_dataset   = ToxicityDataset(eval_enc,   eval_df['label'].tolist())
print(f'Poisoned train: {len(poison_dataset):,}  |  Eval: {len(eval_dataset):,}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score as sk_f1

def compute_metrics(ep):
    logits, labels = ep
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_macro': sk_f1(labels, preds, average='macro')}

print('Initialising fresh distilbert-base-uncased (poisoning attack starts from scratch) …')
model_poison = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_poison.to(DEVICE)

args_poison = TrainingArguments(
    output_dir             = './poisoned_checkpoint',
    num_train_epochs       = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    warmup_ratio           = 0.06,
    weight_decay           = 0.01,
    learning_rate          = 2e-5,
    logging_steps          = 500,
    eval_strategy          = 'epoch',   # renamed from evaluation_strategy in transformers>=4.46
    save_strategy          = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'f1_macro',
    report_to              = 'none',
    fp16                   = torch.cuda.is_available(),
    seed                   = SEED,
)

trainer_poison = Trainer(
    model           = model_poison,
    args            = args_poison,
    train_dataset   = poison_dataset,
    eval_dataset    = eval_dataset,
    compute_metrics = compute_metrics,
)

print('Training poisoned model (3 epochs) …')
trainer_poison.train()
print('Poisoned model training complete.')

In [ ]:
# ── Evaluate poisoned model ───────────────────────────────────────────────────
pred_out_poison = trainer_poison.predict(eval_dataset)
logits_p  = pred_out_poison.predictions
probs_p   = torch.softmax(torch.tensor(logits_p, dtype=torch.float32), dim=-1).numpy()[:, 1]
y_pred_p  = (probs_p >= OPERATING_THRESHOLD).astype(int)
y_true    = eval_df['label'].values

# Metrics — poisoned model
acc_p  = accuracy_score(y_true, y_pred_p)
f1_p   = sk_f1(y_true, y_pred_p, average='macro')
cm_p   = confusion_matrix(y_true, y_pred_p)
tn_p, fp_p, fn_p, tp_p = cm_p.ravel()
fnr_p  = fn_p / (fn_p + tp_p) if (fn_p + tp_p) > 0 else 0
fpr_p  = fp_p / (fp_p + tn_p) if (fp_p + tn_p) > 0 else 0

# Baseline metrics (Part 1 at threshold 0.4)
eval_probs_clean = np.load('eval_probs.npy')
y_pred_clean = (eval_probs_clean >= OPERATING_THRESHOLD).astype(int)
acc_c  = accuracy_score(y_true, y_pred_clean)
f1_c   = sk_f1(y_true, y_pred_clean, average='macro')
cm_c   = confusion_matrix(y_true, y_pred_clean)
tn_c, fp_c, fn_c, tp_c = cm_c.ravel()
fnr_c  = fn_c / (fn_c + tp_c)
fpr_c  = fp_c / (fp_c + tn_c)

print('='*70)
print('  ATTACK 2 RESULTS: Label-Flipping Poisoning (5% label flip)')
print('='*70)
print(f'  {"Metric":<30} {"Baseline (clean)":>18} {"Poisoned model":>18} {"Change":>10}')
print('-'*70)
for label, c_val, p_val in [
    ('Accuracy',           acc_c,  acc_p),
    ('F1 (macro)',         f1_c,   f1_p),
    ('False Negative Rate',fnr_c,  fnr_p),
    ('False Positive Rate',fpr_c,  fpr_p),
]:
    delta = p_val - c_val
    direction = '▲' if delta > 0 else '▼'
    print(f'  {label:<30} {c_val:>18.4f} {p_val:>18.4f} {direction}{abs(delta):>9.4f}')
print('='*70)
print(f'\n  Key finding: FNR increased by {fnr_p-fnr_c:+.4f} — the poisoned model misses')
print(f'  more genuinely toxic comments on the clean test set.')

In [ ]:
# ── Visualise Attack 2 comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of metric changes
metrics_names = ['Accuracy', 'F1 (macro)', 'FNR', 'FPR']
clean_vals    = [acc_c, f1_c, fnr_c, fpr_c]
poison_vals   = [acc_p, f1_p, fnr_p, fpr_p]
x = np.arange(len(metrics_names))
w = 0.35

axes[0].bar(x - w/2, clean_vals,  w, label='Baseline (clean)', color='steelblue', alpha=0.85, edgecolor='black')
axes[0].bar(x + w/2, poison_vals, w, label='Poisoned (5%)',    color='#d62728',   alpha=0.85, edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names, fontsize=11)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Clean vs Poisoned Model (5% label flip)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3, axis='y')

# Confusion matrix comparison
ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred_p),
    display_labels=['Non-Toxic', 'Toxic']
).plot(ax=axes[1], cmap='Reds', colorbar=False)
axes[1].set_title(f'Poisoned Model Confusion Matrix\n(FNR={fnr_p:.3f}, FPR={fpr_p:.3f})',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('part3_attack2.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Question: Which Attack is More Operationally Dangerous?

### Attack 1 — Character-level Evasion
**Threat model**: A bad actor manipulates their *own* comment text before posting.  
**Attacker requirements**: No system access, just knowledge that the classifier exists.  
**Effort per attack**: Moderate — must apply the perturbation to each comment individually.  
**Scale**: Can be automated with a script. Realistically, sophisticated actors (e.g., coordinated
harassment groups) already share evasion techniques.  
**Detectability**: Perturbed text contains non-Latin Unicode characters and zero-width spaces,
which are detectable with a simple character-set filter. A Layer 1 pre-filter (Part 5) can
catch many of these before they reach the model.

### Attack 2 — Label-Flipping Poisoning
**Threat model**: An attacker with write access to the ML training pipeline corrupts training data.  
**Attacker requirements**: Access to the data ingestion system, training scripts, or the
labelling pipeline — a significantly higher bar.  
**Effort per attack**: One-time — corrupting 5% of labels only needs to happen once before
a training run.  
**Effect**: Systematically raises FNR — the poisoned model **consistently misses toxic content**.
This is invisible to end users and very hard to detect without comparing against a clean baseline.  
**Detectability**: Much harder to detect than evasion. Anomalous label distributions could be
caught with data quality checks, but sophisticated attackers can keep flips below detection thresholds.

### Verdict: Which is more realistic for a social platform?

**Attack 1 (evasion) is more commonly attempted** because it requires no system access — any
user can try it. The ASR observed here demonstrates it is effective. However, defenses exist
(Unicode normalization, character-set filtering, adversarial training) and are relatively cheap
to deploy.

**Attack 2 (poisoning) is more catastrophic if it occurs** because it is:
- Hard to detect (the model still appears to work well on most cases)
- Hard to remediate (requires retraining from scratch on clean data)
- Persistent across all subsequent model versions if the poisoned data enters the data lake
- Directionally controlled by the attacker (e.g., only flip toxic→non-toxic for specific hate speech categories)

**Defense prioritization implication**: For most social platforms the near-term priority is
evasion defenses (character normalization, homoglyph detection, adversarial training) because
that is the realistic daily threat. However, poisoning defenses (training data provenance,
anomaly detection on label distributions, data versioning, cryptographic signing of training sets)
are critical **supply-chain** investments for any platform operating at scale, where the
consequences of a successful poisoning attack are far more severe and lasting.